In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    explained_variance_score
)
import optuna
from sklearn.model_selection import cross_val_score

from scipy.stats import pearsonr
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor


In [30]:
print("hello")

hello


In [31]:
data_df = pd.read_csv("D:\APSirCNTandLSTM\CNT with LSTM\Datasets\CNT_Reinforced Concrete_Final Dataset.csv") # Read the data

data_df.dropna(inplace = True) # Drop missing/null values

print("Original Data Shape: ", data_df.shape)

def is_numeric_row(row):
    try:
        pd.to_numeric(row, errors='raise')
        return True
    except:
        return False
    

numeric_rows = data_df.apply(lambda row: is_numeric_row(row), axis=1)
data_df = data_df[numeric_rows]

print("Data Shape (rows, cols): ", data_df.shape) # Print data shape
data_df # Print/visualize the first 5 rows of the data

Original Data Shape:  (295, 7)
Data Shape (rows, cols):  (295, 7)


,Cement (kg/m³),w/c,Coarse aggregate (kg/m³),Fine aggregate (kg/m³),Curing (days),CNT (%),Compressive strength(MPa)
0,360.0,0.50,950.0,745.0,28,0.00,39.500
1,320.0,0.56,898.0,782.0,28,0.00,40.000
2,328.5,0.68,533.7,625.1,28,0.00,40.000
3,360.0,0.50,898.0,797.0,28,0.00,40.000
4,320.0,0.56,845.0,834.0,28,0.00,40.300
...,...,...,...,...,...,...,...
290,400.0,0.45,987.0,780.0,28,0.05,64.000
291,450.0,0.35,1006.0,768.0,28,0.05,84.900
292,525.0,0.45,0.0,1580.0,28,0.00,29.875
293,525.0,0.45,0.0,1580.0,28,0.06,38.750


In [32]:
#Converting non-numeric columns to numeric, if any. This will convert non-numeric values to NaN, which will be dropped later.

for column in data_df.columns:
    data_df[column] = pd.to_numeric(data_df[column], errors='coerce')


target_column = "Compressive strength(MPa)"

# Check data types
print("Data types:")
print(data_df.dtypes)
print("\n" + "="*50 + "\n")

# Check if target column has issues
print(f"Target column '{target_column}' unique values (first 10):")
print(data_df[target_column].unique()[:10])
print(f"\nTarget column type: {data_df[target_column].dtype}")

# Check for any remaining object columns
object_cols = data_df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject (string) columns: {object_cols}")

Data types:
Cement (kg/m³)               float64
w/c                          float64
Coarse aggregate (kg/m³)     float64
Fine aggregate (kg/m³)       float64
Curing (days)                  int64
CNT (%)                      float64
Compressive strength(MPa)    float64
dtype: object


Target column 'Compressive strength(MPa)' unique values (first 10):
[39.5 40.  40.3 40.8 41.  41.1 41.3 41.5 41.8 42. ]

Target column type: float64

Object (string) columns: []


In [33]:
# Prepare Input and Output

X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)


Input Shape: (295, 6)
Output Shape: (295,)


In [34]:
X, Y

(array([[3.600e+02, 5.000e-01, 9.500e+02, 7.450e+02, 2.800e+01, 0.000e+00],
        [3.200e+02, 5.600e-01, 8.980e+02, 7.820e+02, 2.800e+01, 0.000e+00],
        [3.285e+02, 6.800e-01, 5.337e+02, 6.251e+02, 2.800e+01, 0.000e+00],
        ...,
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 0.000e+00],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 6.000e-02],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 1.200e-01]]),
 array([39.5  , 40.   , 40.   , 40.   , 40.3  , 40.8  , 40.8  , 41.   ,
        41.1  , 41.3  , 41.3  , 41.5  , 41.5  , 41.5  , 41.5  , 41.8  ,
        42.   , 42.1  , 42.4  , 42.5  , 43.   , 43.2  , 43.3  , 43.5  ,
        43.6  , 43.8  , 44.   , 44.5  , 45.   , 45.   , 45.1  , 45.7  ,
        46.   , 46.   , 47.1  , 47.7  , 50.   , 36.8  , 43.1  , 38.6  ,
        47.2  , 45.1  , 49.6  , 47.4  , 54.   , 50.1  , 37.8  , 44.1  ,
        40.9  , 47.5  , 45.3  , 42.5  , 52.   , 48.7  , 46.6  , 54.5  ,
        53.1  , 49.2  , 4

In [35]:

# 80 : 20

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

print("Training X:", X_train.shape)
print("Testing X:", X_test.shape)

print("Training y:", y_train.shape)
print("Testing y:", y_test.shape)


print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%") # Print training shape
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%") # Print testing shape


Training X: (236, 6)
Testing X: (59, 6)
Training y: (236,)
Testing y: (59,)
Training set is:  236  rows which is  80.0 %
Testing set is:  59  rows which is  20.0 %


In [36]:
scaler_X_XGB = StandardScaler()

X_train_scaled = scaler_X_XGB.fit_transform(X_train)
X_test_scaled = scaler_X_XGB.transform(X_test)

print("XGB Training Shape X:", X_train_scaled.shape)
print("XGB testing Shape X:", X_test_scaled.shape)
print("**********************************")
print("XGB Training Shape y:", y_train.shape)
print("XGB testing Shape y:", y_test.shape)


XGB Training Shape X: (236, 6)
XGB testing Shape X: (59, 6)
**********************************
XGB Training Shape y: (236,)
XGB testing Shape y: (59,)


In [37]:
# Model
knn_model = KNeighborsRegressor(
    n_neighbors=5,
    weights='distance',      # or 'uniform'
    metric='minkowski',      # Euclidean distance when p=2
    p=2
)

# Training
knn_model.fit(
    X_train_scaled,
    y_train
)

# Prediction
y_pred_train = knn_model.predict(X_train_scaled)

y_pred_test = knn_model.predict(X_test_scaled)

print(y_pred_test)
print(y_pred_train)

[57.42421688 59.45332296 52.4018275  50.00344832 37.62162162 38.50356195
 52.         49.74312407 49.68657897 53.4        52.9        30.5435896
 48.86899814 51.61       58.99675676 36.8        50.29629083 35.77666667
 19.775      52.7672206  62.6        58.76206434 54.90183785 58.7
 43.5        42.32014893 38.4372973  20.85081081 43.9        37.18794808
 21.42965026 46.12193601 41.6596794  41.6596794  37.99532998 40.7
 58.07293938 53.13190027 49.5        32.64       23.81       52.99815854
 54.18913596 51.45972674 45.61187712 43.56       45.26442161 47.36277013
 41.10605946 53.827193   50.         59.2        42.1173704  41.36857486
 17.47216216 42.48745722 50.29629083 44.5        53.13190027]
[46.05       39.5        42.1        45.3        19.775      59.5
 48.2        53.9        47.66666667 40.7        54.8        62.6
 49.75       42.5        50.4        39.6        57.6        34.15
 41.8        41.3        55.4        45.55       58.15       36.8
 42.         29.875      52.9  

In [38]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    # avoid divide-by-zero
    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    metrics = {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }

    return metrics

print("TRAIN METRICS")
print(evaluate_regression(y_train, y_pred_train))


print("\nTEST METRICS")
print(evaluate_regression(y_test, y_pred_test))

TRAIN METRICS
{'R2': 0.9955449679237812, 'MSE': 0.48027718926553686, 'RMSE': 0.6930203382769778, 'MAE': 0.2827824858757062, 'MAPE (%)': 0.6157103380254701, 'Explained Variance Score': 0.9955449679237812, 'Pearson Correlation': 0.9977699975063293, 'Relative RMSE (%)': 1.5182993466306425}

TEST METRICS
{'R2': 0.7778717634039926, 'MSE': 32.24178780750157, 'RMSE': 5.678185256532369, 'MAE': 3.26848810485085, 'MAPE (%)': 10.924693025765773, 'Explained Variance Score': 0.7888202132441645, 'Pearson Correlation': 0.888346975036746, 'Relative RMSE (%)': 12.845440049976794}


In [39]:
import pandas as pd

train_metrics = evaluate_regression(y_train, y_pred_train)
test_metrics  = evaluate_regression(y_test, y_pred_test)

df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})

print(df_results.round(4))

                           Train     Test
R2                        0.9955   0.7779
MSE                       0.4803  32.2418
RMSE                      0.6930   5.6782
MAE                       0.2828   3.2685
MAPE (%)                  0.6157  10.9247
Explained Variance Score  0.9955   0.7888
Pearson Correlation       0.9978   0.8883
Relative RMSE (%)         1.5183  12.8454


In [40]:

# OPTUNA + XGBOOST + 5-FOLD CV

# K-Fold

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective function
def objective(trial):

    params = {

        "n_neighbors": trial.suggest_int(
            "n_neighbors",
            2,
            30
        ),

        "weights": trial.suggest_categorical(
            "weights",
            ["uniform", "distance"]
        ),

        "p": trial.suggest_int(
            "p",
            1,
            2
        ),

        "metric": "minkowski"
    }

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(**params))
    ])

    scores = cross_val_score(
        pipeline,
        X_train,      # unscaled data
        y_train,
        cv=kfold,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()



In [41]:
study = optuna.create_study(
    direction="maximize",
    study_name="KNN_study",
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-03 15:54:05,463] A new study created in memory with name: KNN_study
[I 2026-06-03 15:54:05,480] Trial 0 finished with value: 0.6848848777329559 and parameters: {'n_neighbors': 3, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.6848848777329559.
[I 2026-06-03 15:54:05,495] Trial 1 finished with value: 0.5572032843115597 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.6848848777329559.
[I 2026-06-03 15:54:05,510] Trial 2 finished with value: 0.7019335490071948 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'p': 2}. Best is trial 2 with value: 0.7019335490071948.
[I 2026-06-03 15:54:05,523] Trial 3 finished with value: 0.6423470950558677 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'p': 1}. Best is trial 2 with value: 0.7019335490071948.
[I 2026-06-03 15:54:05,537] Trial 4 finished with value: 0.5538919742390619 and parameters: {'n_neighbors': 12, 'weights': 'uniform', 'p': 2}. Best is trial 2

In [42]:
print("Best Parameters:")
print(study.best_params)

print("\nBest CV R2:")
print(study.best_value)

Best Parameters:
{'n_neighbors': 6, 'weights': 'distance', 'p': 1}

Best CV R2:
0.7597452881299669


In [43]:
best_knn = KNeighborsRegressor(
    **study.best_params
)

best_knn.fit(
    X_train_scaled,
    y_train
)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",1
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [51]:
y_pred_train = best_knn.predict(X_train_scaled)
y_pred_test  = best_knn.predict(X_test_scaled)

In [52]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }


train_metrics = evaluate_regression(y_train, y_pred_train)


test_metrics = evaluate_regression(y_test, y_pred_test)



df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})


print(df_results.round(4))



                           Train     Test
R2                        0.9955   0.8033
MSE                       0.4803  28.5538
RMSE                      0.6930   5.3436
MAE                       0.2828   3.1222
MAPE (%)                  0.6157  10.6535
Explained Variance Score  0.9955   0.8155
Pearson Correlation       0.9978   0.9056
Relative RMSE (%)         1.5183  12.0885


In [53]:
#Save XGBoost model using pickle

import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\KNN\Saved models\KNN_model8020.pkl", "wb") as f:
    pickle.dump(best_knn, f)

print("KNN model saved successfully.")

KNN model saved successfully.


In [54]:
import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\KNN\Saved models\KNN_scalar8020.pkl", "wb") as f:
    pickle.dump(scaler_X_XGB, f)

print("KNN scalar saved successfully.")

KNN scalar saved successfully.


In [55]:
#Load model back

import pickle

with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\KNN\Saved models\KNN_model8020.pkl", "rb") as f:
    loaded_svr = pickle.load(f)

print("Model loaded successfully.")

Model loaded successfully.


In [56]:
#Recompute predictions (train / val / test)

y_pred_train_loaded = loaded_svr.predict(X_train_scaled)
y_pred_test_loaded  = loaded_svr.predict(X_test_scaled)

In [57]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    explained_variance_score
)

from scipy.stats import pearsonr


def evaluate(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs = explained_variance_score(y_true, y_pred)
    pearson_r, _ = pearsonr(y_true, y_pred)
    rrmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": rrmse
    }

train_metrics_loaded = evaluate(y_train, y_pred_train_loaded)
test_metrics_loaded  = evaluate(y_test, y_pred_test_loaded)


results_df = pd.DataFrame({
    "Train": train_metrics_loaded,
    "Test": test_metrics_loaded
})

print(results_df.round(4))

                      Train     Test
R2                   0.9955   0.8033
MSE                  0.4803  28.5538
RMSE                 0.6930   5.3436
MAE                  0.2828   3.1222
MAPE (%)             0.6157  10.6535
Explained Variance   0.9955   0.8155
Pearson Correlation  0.9978   0.9056
Relative RMSE (%)    1.5183  12.0885
